# Condition estimation with naval propulsion data

Condition estimation predicts a continuous health quantity. CBM provides
compressor and turbine degradation coefficients as explicit regression targets.

## Learning goals

- distinguish condition estimation from discrete fault classification;
- train a multivariate regression baseline;
- inspect absolute error across the degradation range.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import datasets
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

In [ ]:
datasets.download("cbm")
data = datasets.load("cbm")
target = "kMc"
features = [column for column in data.columns if column not in {"kMc", "kMt"}]
data.select(features + [target]).describe()

In [ ]:
indices = np.arange(data.height)
train_index, test_index = train_test_split(indices, test_size=0.2, random_state=0)
model = ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1)
model.fit(data[train_index].select(features).to_numpy(), data[train_index][target].to_numpy())
prediction = model.predict(data[test_index].select(features).to_numpy())
actual = data[test_index][target].to_numpy()
print(f"Compressor-degradation MAE: {mean_absolute_error(actual, prediction):.5f}")

In [ ]:
plt.scatter(actual, prediction, s=8, alpha=0.4)
limits = [min(actual.min(), prediction.min()), max(actual.max(), prediction.max())]
plt.plot(limits, limits, "--", color="black")
plt.xlabel("Actual compressor degradation coefficient")
plt.ylabel("Predicted coefficient")
plt.show()

## Interpretation and limitations

CBM is simulated steady-state coverage rather than a run-to-failure sequence.
It supports condition regression, but not event timing or survival analysis.
Random splitting measures interpolation and should not be described as
cross-vessel or temporal generalization.